In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import glob
import os
import onnxruntime as ort  # Pour exécuter le modèle quantifié ONNX

# =============================================================================
# --- 1. CONFIGURATION ET CHARGEMENT DU MODÈLE ONNX QUANTIFIÉ (W16A8) ---
# =============================================================================

checkpoint_dir = "../models_FFNN"
search_pattern = os.path.join(checkpoint_dir, "*.onnx")
list_of_files = glob.glob(search_pattern)

if not list_of_files:
    raise FileNotFoundError(f"⚠️ Aucun fichier .onnx trouvé dans {checkpoint_dir}")

# Sélection du fichier ONNX le plus récent
latest_model_path = max(list_of_files, key=os.path.getctime)

# Initialisation de la session ONNX Runtime (CPU/GPU)
try:
    ort_session = ort.InferenceSession(latest_model_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
except Exception:
    ort_session = ort.InferenceSession(latest_model_path, providers=['CPUExecutionProvider'])

input_name = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name

print(f"✅ MODÈLE ONNX W16A8 CHARGÉ : {os.path.basename(latest_model_path)}")

# --- Importation de ta fonction MVDR théorique ---
from MVDR import mvdr_results 

# =============================================================================
# --- 2. CONFIGURATION DE LA TRIADE UNIQUE ET DES PARAMÈTRES RADAR ---
# =============================================================================

# MODIFIE CES VALEURS POUR TESTER LES ANGLES QUE TU VEUX :
soi_test  = 0.0    # Angle du signal utile (SOI) en degrés
soa1_test = -30.0  # Angle du premier brouilleur (SOA1) en degrés
soa2_test = 45.0   # Angle du deuxième brouilleur (SOA2) en degrés

Nr = 16  # Nombre d'éléments du réseau linéaire
d = 0.5  # Espacement des antennes (normalisé par la longueur d'onde)

# Scans d'angles pour le tracé du diagramme (-90° à +90°)
theta_scan_deg = np.linspace(-90, 90, 1000)

# =============================================================================
# --- 3. FONCTION UTILITAIRE (VECTEUR DIRECTEUR) ---
# =============================================================================
def get_steering_vector(theta_deg, Nr=16, d=0.5):
    theta_rad = np.deg2rad(theta_deg)
    n = np.arange(Nr)
    return np.exp(2j * np.pi * d * n * np.sin(theta_rad))

def get_gain_at_angle(angle_cible, angles_scan, gains_db):
    idx_proche = np.argmin(np.abs(angles_scan - angle_cible))
    return gains_db[idx_proche]

# =============================================================================
# --- 4. INFERENCE DU MODÈLE ET CALCUL MVDR ---
# =============================================================================

# --- A. Prédiction du MLP Quantifié (W16A8) ---
# Normalisation d'entrée (/60.0) comme à l'entraînement
input_data = np.array([[soi_test, soa1_test, soa2_test]], dtype=np.float32) / 60.0
onnx_outputs = ort_session.run([output_name], {input_name: input_data})
sortie = onnx_outputs[0].flatten()

# Dé-normalisation (*10) et reconstruction des poids complexes (16 Réels, 16 Imaginaires)
sortie_norm = sortie * 10   
w_nn = sortie_norm[:16] + 1j * sortie_norm[16:]

# --- B. Calcul MVDR Théorique ---
w_mvdr, _ = mvdr_results(soi_test, soa1_test, soa2_test)
w_mvdr = w_mvdr.flatten()

# =============================================================================
# --- 5. CALCUL DES DIAGRAMMES DE RAYONNEMENT ---
# =============================================================================

# Calcul du gain du modèle quantifié sur toute la plage [-90°, 90°]
p_nn = np.array([np.abs(np.vdot(w_nn, get_steering_vector(ang)))**2 for ang in theta_scan_deg])
ref_nn = np.abs(np.vdot(w_nn, get_steering_vector(soi_test)))**2
db_nn_plot = 10 * np.log10(p_nn / ref_nn + 1e-12)

# Calcul du gain MVDR théorique sur toute la plage [-90°, 90°]
p_mv = np.array([np.abs(np.vdot(w_mvdr, get_steering_vector(ang)))**2 for ang in theta_scan_deg])
ref_mv = np.abs(np.vdot(w_mvdr, get_steering_vector(soi_test)))**2
db_mv_plot = 10 * np.log10(p_mv / ref_mv + 1e-12)



In [ ]:
# =============================================================================
# --- 6. TRACÉ UNIQUE DU GRAPHIQUE ---
# =============================================================================
plt.figure(figsize=(12, 6))

# Tracé des courbes de réponse spatiale
plt.plot(theta_scan_deg, db_mv_plot, color='red', label='MVDR théorique', linewidth=2)
plt.plot(theta_scan_deg, db_nn_plot, color='blue', linestyle='--', label='MLP Quantifié W16A8 (ONNX)', linewidth=1.8)

# Balisage et affichage des valeurs aux angles clés (SOI, SOA1, SOA2)
triade_angles = [soi_test, soa1_test, soa2_test]
couleurs = ['green', 'orange', 'orange']
labels = ['SOI', 'Brouilleur 1', 'Brouilleur 2']

for j, ang in enumerate(triade_angles):
    g_nn = get_gain_at_angle(ang, theta_scan_deg, db_nn_plot)
    g_mv = get_gain_at_angle(ang, theta_scan_deg, db_mv_plot)
    
    # Ligne verticale en pointillés pour l'angle
    plt.axvline(ang, color=couleurs[j], alpha=0.7, linestyle=':', linewidth=1.5)
    
    # Point bleu sur la courbe du réseau de neurones quantifié
    plt.plot(ang, g_nn, 'bo', markersize=6)
    
    # Étiquette texte
    plt.annotate(f"{labels[j]} : {ang:.1f}°\nNN: {g_nn:.1f} dB\nMV: {g_mv:.1f} dB", 
                 (ang, g_nn), xytext=(15, -20), 
                 textcoords="offset points", ha='left', fontsize=9, 
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8, ec='gray'))

# Mise en forme du graphique pour le rapport
plt.title(f"Diagramme de Rayonnement | SOI: {soi_test}° | SOA1: {soa1_test}° | SOA2: {soa2_test}°", fontsize=12)
plt.xlabel("Angle de balayage (degrés)", fontsize=10)
plt.ylabel("Gain Spatiale Relatif (dB)", fontsize=10)
plt.xlim([-90, 90])
plt.xticks(range(-90, 91, 10))
plt.ylim([-90, 5])
plt.yticks(range(-90, 11, 10))
plt.grid(True, linestyle='-', alpha=0.4)
plt.legend(loc='upper right', fontsize='medium')

plt.tight_layout()
plt.show()